In [ ]:
#CELL 1: Imports and paths
import pandas as pd
import numpy as np
import os

RAW       = "../data/raw"
PROCESSED = "../data/processed"

os.makedirs(PROCESSED, exist_ok=True)

# Confirm files exist before proceeding
for fname in ["LLCP2022.XPT", "LLCP2023.XPT"]:
    path = os.path.join(RAW, fname)
    exists = os.path.exists(path)
    size   = f"{os.path.getsize(path)/1e6:.1f} MB" if exists else "—"
    print(f"  {'✓' if exists else '✗'}  {fname}  ({size})")

  ✓  LLCP2022.XPT  (1160.1 MB)
  ✓  LLCP2023.XPT  (1205.6 MB)


In [ ]:
# CELL 2: Define variables to keep

KEEP_VARS = {

    # Outcome
    "CHCCOPD3":  "COPD/emphysema/chronic bronchitis (1=Yes, 2=No)",

    # Demographics & SES
    "INCOME3":   "Annual household income (1–11 scale, higher=more)",
    "EDUCA":     "Education level (1–6)",
    "SEXVAR":    "Sex (1=Male, 2=Female)",
    "_AGEG5YR":  "Age group in 5-year bands (1–13)",
    "_IMPRACE":  "Race/ethnicity imputed (1–6)",
    "EMPLOY1":   "Employment status (1–8)",
    "MARITAL":   "Marital status (1–6)",
    "_URBSTAT":  "Urban/rural status (1=Urban, 2=Rural)",

    # Behavioral
    "_SMOKER3":  "Smoking status (1=Current daily, 2=Current someday, 3=Former, 4=Never)",
    "EXERANY2":  "Any physical activity past 30 days (1=Yes, 2=No)",
    "DRNKANY6":  "Any alcohol in past 30 days (1=Yes, 2=No)",
    "_BMI5CAT":  "BMI category (1=Underweight, 2=Normal, 3=Overweight, 4=Obese)",

    # Comorbidities (core — all states)
    "ASTHMA3":   "Ever told had asthma (1=Yes, 2=No)",
    "CVDINFR4":  "Ever told had heart attack (1=Yes, 2=No)",
    "CVDCRHD4":  "Ever told had coronary heart disease (1=Yes, 2=No)",
    "CVDSTRK3":  "Ever told had stroke (1=Yes, 2=No)",
    "DIABETE4":  "Ever told had diabetes (1=Yes, 2=No, 3=Pre-diabetes, 4=Gestational)",
    "HAVARTH3":  "Ever told had arthritis (1=Yes, 2=No)",
    #both BP high and ToldHi are not in this dataset
    "CHCKDNY2":  "Ever told had kidney disease (1=Yes, 2=No)",

    # Access to care
    "HLTHPLN2":  "Have any health insurance (1=Yes, 2=No)",
    "PERSDOC3":  "Have personal doctor (1=Yes one, 2=Yes more than one, 3=No)",
    "MEDCOST1":  "Could not see doctor due to cost (1=Yes, 2=No)",
    "CHECKUP1":  "Last routine checkup (1=<1yr, 2=1-2yr, 3=2-5yr, 4=>5yr, 8=Never)",

    # Survey design variables (needed for proper weighting)
    "_LLCPWT":   "Final combined weight",
    "_STSTR":    "Stratum",
    "_PSU":      "Primary sampling unit",
}

print(f"Variables to extract: {len(KEEP_VARS)}")
for var, desc in KEEP_VARS.items():
    print(f"  {var:<12} {desc}")

Variables to extract: 27
  CHCCOPD3     COPD/emphysema/chronic bronchitis (1=Yes, 2=No)
  INCOME3      Annual household income (1–11 scale, higher=more)
  EDUCA        Education level (1–6)
  SEXVAR       Sex (1=Male, 2=Female)
  _AGEG5YR     Age group in 5-year bands (1–13)
  _IMPRACE     Race/ethnicity imputed (1–6)
  EMPLOY1      Employment status (1–8)
  MARITAL      Marital status (1–6)
  _URBSTAT     Urban/rural status (1=Urban, 2=Rural)
  _SMOKER3     Smoking status (1=Current daily, 2=Current someday, 3=Former, 4=Never)
  EXERANY2     Any physical activity past 30 days (1=Yes, 2=No)
  DRNKANY6     Any alcohol in past 30 days (1=Yes, 2=No)
  _BMI5CAT     BMI category (1=Underweight, 2=Normal, 3=Overweight, 4=Obese)
  ASTHMA3      Ever told had asthma (1=Yes, 2=No)
  CVDINFR4     Ever told had heart attack (1=Yes, 2=No)
  CVDCRHD4     Ever told had coronary heart disease (1=Yes, 2=No)
  CVDSTRK3     Ever told had stroke (1=Yes, 2=No)
  DIABETE4     Ever told had diabetes (1=Yes, 

In [ ]:
#CELL 3: Load raw BRFSS files 


print("Loading BRFSS 2022...")
raw_2022 = pd.read_sas(
    os.path.join(RAW, "LLCP2022.XPT"),
    format="xport",
    encoding="latin-1"
)
raw_2022.columns = raw_2022.columns.str.upper()
print(f"  2022: {raw_2022.shape[0]:,} rows × {raw_2022.shape[1]:,} columns")

print("Loading BRFSS 2023...")
raw_2023 = pd.read_sas(
    os.path.join(RAW, "LLCP2023.XPT"),
    format="xport",
    encoding="latin-1"
)
raw_2023.columns = raw_2023.columns.str.upper()
print(f"  2023: {raw_2023.shape[0]:,} rows × {raw_2023.shape[1]:,} columns")

Loading BRFSS 2022...
  2022: 445,132 rows × 328 columns
Loading BRFSS 2023...
  2023: 433,323 rows × 350 columns


In [ ]:
# CELL 4: Check which variables are actually present

print("=== Variable availability check ===\n")
print(f"{'Variable':<14} {'2022':>6} {'2023':>6}  Description")
print("-" * 70)

missing = {"2022": [], "2023": []}

for var, desc in KEEP_VARS.items():
    in_22 = var in raw_2022.columns
    in_23 = var in raw_2023.columns
    flag = "  ⚠" if not (in_22 and in_23) else ""
    print(f"  {var:<12} {'✓' if in_22 else '✗':>6} {'✓' if in_23 else '✗':>6}  {desc[:45]}{flag}")
    if not in_22: missing["2022"].append(var)
    if not in_23: missing["2023"].append(var)

print(f"\nMissing in 2022: {missing['2022'] if missing['2022'] else 'None'}")
print(f"Missing in 2023: {missing['2023'] if missing['2023'] else 'None'}")
print("\n⚠ If any variables are missing, check the BRFSS codebook for")
print("  the correct name in that year before proceeding to Cell 5.")

=== Variable availability check ===

Variable         2022   2023  Description
----------------------------------------------------------------------
  CHCCOPD3          ✓      ✓  COPD/emphysema/chronic bronchitis (1=Yes, 2=N
  INCOME3           ✓      ✓  Annual household income (1–11 scale, higher=m
  EDUCA             ✓      ✓  Education level (1–6)
  SEXVAR            ✓      ✓  Sex (1=Male, 2=Female)
  _AGEG5YR          ✓      ✓  Age group in 5-year bands (1–13)
  _IMPRACE          ✓      ✓  Race/ethnicity imputed (1–6)
  EMPLOY1           ✓      ✓  Employment status (1–8)
  MARITAL           ✓      ✓  Marital status (1–6)
  _URBSTAT          ✓      ✓  Urban/rural status (1=Urban, 2=Rural)
  _SMOKER3          ✓      ✓  Smoking status (1=Current daily, 2=Current so
  EXERANY2          ✓      ✓  Any physical activity past 30 days (1=Yes, 2=
  DRNKANY6          ✓      ✓  Any alcohol in past 30 days (1=Yes, 2=No)
  _BMI5CAT          ✓      ✓  BMI category (1=Underweight, 2=Normal, 3=Ove

In [ ]:
# CELL 4b: Resolve missing variables — rename + fix inverted coding

print("=== Resolving missing variables ===\n")

# ── 1. Blood pressure ─────────────────────────────────────────────────
if "BPHIGH6" not in raw_2022.columns and "_RFHYPE6" in raw_2022.columns:
    raw_2022 = raw_2022.rename(columns={"_RFHYPE6": "BPHIGH6"})
    # _RFHYPE6: 1=No, 2=Yes → flip to 1=Yes, 2=No to match BPHIGH6 coding
    raw_2022["BPHIGH6"] = raw_2022["BPHIGH6"].map({1.0: 2.0, 2.0: 1.0, 9.0: np.nan})
    print("  BPHIGH6 (2022): renamed from _RFHYPE6 + inverted 1/2 coding ✓")
elif "BPHIGH6" in raw_2022.columns:
    print("  BPHIGH6 (2022): already present ✓")
else:
    print("  BPHIGH6 (2022): NOT FOUND — will be excluded ✗")

# ── 2. Cholesterol ────────────────────────────────────────────────────
if "TOLDHI3" not in raw_2022.columns and "_RFCHOL3" in raw_2022.columns:
    raw_2022 = raw_2022.rename(columns={"_RFCHOL3": "TOLDHI3"})
    # _RFCHOL3: 1=No, 2=Yes → flip to 1=Yes, 2=No to match TOLDHI3 coding
    raw_2022["TOLDHI3"] = raw_2022["TOLDHI3"].map({1.0: 2.0, 2.0: 1.0, 9.0: np.nan})
    print("  TOLDHI3 (2022): renamed from _RFCHOL3 + inverted 1/2 coding ✓")
elif "TOLDHI3" in raw_2022.columns:
    print("  TOLDHI3 (2022): already present ✓")
else:
    print("  TOLDHI3 (2022): NOT FOUND — will be excluded ✗")

# ── 3. Health insurance ───────────────────────────────────────────────
# 2022: _HLTHPLN (1=Yes, 2=No) — coding already matches target HLTHPLN2
# 2023: _HLTHPL1 (1=Yes, 2=No) — coding already matches target HLTHPLN2
if "HLTHPLN2" not in raw_2022.columns and "_HLTHPLN" in raw_2022.columns:
    raw_2022 = raw_2022.rename(columns={"_HLTHPLN": "HLTHPLN2"})
    print("  HLTHPLN2 (2022): renamed from _HLTHPLN, no coding fix needed ✓")
elif "HLTHPLN2" in raw_2022.columns:
    print("  HLTHPLN2 (2022): already present ✓")
else:
    print("  HLTHPLN2 (2022): NOT FOUND — will be excluded ✗")

if "HLTHPLN2" not in raw_2023.columns and "_HLTHPL1" in raw_2023.columns:
    raw_2023 = raw_2023.rename(columns={"_HLTHPL1": "HLTHPLN2"})
    print("  HLTHPLN2 (2023): renamed from _HLTHPL1, no coding fix needed ✓")
elif "HLTHPLN2" in raw_2023.columns:
    print("  HLTHPLN2 (2023): already present ✓")
else:
    print("  HLTHPLN2 (2023): NOT FOUND — will be excluded ✗")

# ── 4. Arthritis (already confirmed in previous run) ─────────────────
if "HAVARTH3" not in raw_2022.columns and "HAVARTH4" in raw_2022.columns:
    raw_2022 = raw_2022.rename(columns={"HAVARTH4": "HAVARTH3"})
    print("  HAVARTH3 (2022): renamed from HAVARTH4 ✓")
if "HAVARTH3" not in raw_2023.columns and "HAVARTH4" in raw_2023.columns:
    raw_2023 = raw_2023.rename(columns={"HAVARTH4": "HAVARTH3"})
    print("  HAVARTH3 (2023): renamed from HAVARTH4 ✓")

# ── 5. Re-run availability check ─────────────────────────────────────
print()
missing = {"2022": [], "2023": []}
print(f"{'Variable':<14} {'2022':>6} {'2023':>6}  Description")
print("-" * 70)
for var, desc in KEEP_VARS.items():
    in_22 = var in raw_2022.columns
    in_23 = var in raw_2023.columns
    flag = "  ⚠ STILL MISSING" if not (in_22 and in_23) else ""
    print(f"  {var:<12} {'✓' if in_22 else '✗':>6} {'✓' if in_23 else '✗':>6}  {desc[:45]}{flag}")
    if not in_22: missing["2022"].append(var)
    if not in_23: missing["2023"].append(var)

print(f"\nStill missing in 2022: {missing['2022'] if missing['2022'] else 'None'}")
print(f"Still missing in 2023: {missing['2023'] if missing['2023'] else 'None'}")

=== Resolving missing variables ===

  BPHIGH6 (2022): NOT FOUND — will be excluded ✗
  TOLDHI3 (2022): NOT FOUND — will be excluded ✗
  HLTHPLN2 (2022): renamed from _HLTHPLN, no coding fix needed ✓
  HLTHPLN2 (2023): renamed from _HLTHPL1, no coding fix needed ✓
  HAVARTH3 (2022): renamed from HAVARTH4 ✓
  HAVARTH3 (2023): renamed from HAVARTH4 ✓

Variable         2022   2023  Description
----------------------------------------------------------------------
  CHCCOPD3          ✓      ✓  COPD/emphysema/chronic bronchitis (1=Yes, 2=N
  INCOME3           ✓      ✓  Annual household income (1–11 scale, higher=m
  EDUCA             ✓      ✓  Education level (1–6)
  SEXVAR            ✓      ✓  Sex (1=Male, 2=Female)
  _AGEG5YR          ✓      ✓  Age group in 5-year bands (1–13)
  _IMPRACE          ✓      ✓  Race/ethnicity imputed (1–6)
  EMPLOY1           ✓      ✓  Employment status (1–8)
  MARITAL           ✓      ✓  Marital status (1–6)
  _URBSTAT          ✓      ✓  Urban/rural status (1

In [ ]:
# CELL 5: Select and rename variables 

available = [v for v in KEEP_VARS if v in raw_2022.columns and v in raw_2023.columns]
print(f"Selecting {len(available)} variables present in both years")

df_2022 = raw_2022[available].copy()
df_2023 = raw_2023[available].copy()

df_2022["YEAR"] = 2022
df_2023["YEAR"] = 2023

print(f"2022: {df_2022.shape}")
print(f"2023: {df_2023.shape}")

Selecting 27 variables present in both years
2022: (445132, 28)
2023: (433323, 28)


In [ ]:
#CELL 6: Recode COPD outcome 

def recode_copd(df, label):
    before = len(df)
    df = df[df["CHCCOPD3"].isin([1.0, 2.0])].copy()
    df["COPD"] = (df["CHCCOPD3"] == 1.0).astype(int)
    after = len(df)
    prevalence = df["COPD"].mean() * 100
    print(f"  {label}: {before:,} → {after:,} rows  |  "
          f"COPD prevalence: {prevalence:.1f}%  "
          f"(N={df['COPD'].sum():,} cases)")
    return df

df_2022 = recode_copd(df_2022, "2022")
df_2023 = recode_copd(df_2023, "2023")

  2022: 445,132 → 442,913 rows  |  COPD prevalence: 8.1%  (N=35,656 cases)
  2023: 433,323 → 431,257 rows  |  COPD prevalence: 7.7%  (N=33,097 cases)


In [ ]:
#CELL 7: Recode features 

# Variable-specific refuse codes (defined outside function, used inside)
GENERIC_REFUSE      = [7, 9, 77, 99, 7.0, 9.0, 77.0, 99.0]
GENERIC_REFUSE_VARS = [
    "SEXVAR", "EDUCA", "_IMPRACE", "MARITAL", "_URBSTAT",
    "_SMOKER3", "_BMI5CAT", "EXERANY2", "DRNKANY6",
    "ASTHMA3", "CVDINFR4", "CVDCRHD4", "CVDSTRK3",
    "HAVARTH3", "CHCKDNY2", "HLTHPLN2", "PERSDOC3",
    "MEDCOST1", "CHECKUP1", "DIABETE4",
]

def clean_features(df):
    df = df.copy()

    # ── Step 1: Variable-specific refuse/DK replacement ──────────────────
    # Blanket 7/9 replacement only on variables where max valid code < 7
    for col in GENERIC_REFUSE_VARS:
        if col in df.columns:
            df[col] = df[col].replace(GENERIC_REFUSE, np.nan)

    # INCOME3: valid 1–11, refuse = 77 (DK) and 99 (Refused) only
    if "INCOME3" in df.columns:
        df["INCOME3"] = df["INCOME3"].replace([77, 77.0, 99, 99.0], np.nan)

    # EMPLOY1: valid 1–8, refuse = 9 only. Value 7 = Retired — keep it.
    if "EMPLOY1" in df.columns:
        df["EMPLOY1"] = df["EMPLOY1"].replace([9, 9.0], np.nan)

    # _AGEG5YR: valid 1–13, refuse = 14 only. Values 7 and 9 are valid.
    if "_AGEG5YR" in df.columns:
        df["_AGEG5YR"] = df["_AGEG5YR"].replace([14, 14.0], np.nan)

    # ── Step 2: Binary yes/no → 1=Yes, 0=No ──────────────────────────────
    yes_no_vars = ["EXERANY2", "DRNKANY6", "HLTHPLN2", "MEDCOST1",
                   "ASTHMA3", "CVDINFR4", "CVDCRHD4", "CVDSTRK3",
                   "HAVARTH3", "CHCKDNY2"]
    for col in yes_no_vars:
        if col in df.columns:
            df[col] = df[col].map({1.0: 1, 2.0: 0})

    # ── Step 3: Blood pressure ────────────────────────────────────────────
    # 1=Yes, 2=No, 3=Borderline → 0, 4=Pregnancy only → 0
    if "BPHIGH6" in df.columns:
        df["BPHIGH6"] = df["BPHIGH6"].map({1.0: 1, 2.0: 0, 3.0: 0, 4.0: 0})

    # ── Step 4: Diabetes ──────────────────────────────────────────────────
    # DIABETE4 coding: 1=Yes, 2=Yes gestational, 3=No, 4=Pre-diabetes
    if "DIABETE4" in df.columns:
        df["DIABETES"]    = df["DIABETE4"].map({1.0: 1, 2.0: 0, 3.0: 0, 4.0: 0})
        df["PREDIABETES"] = df["DIABETE4"].map({1.0: 0, 2.0: 0, 3.0: 0, 4.0: 1})
        df = df.drop(columns=["DIABETE4"])

    # ── Step 5: Cholesterol ───────────────────────────────────────────────
    # 3 = "Not checked" — genuinely missing, not No
    if "TOLDHI3" in df.columns:
        df["TOLDHI3"] = df["TOLDHI3"].map({1.0: 1, 2.0: 0, 3.0: np.nan})

    return df

df_2022 = clean_features(df_2022)
df_2023 = clean_features(df_2023)

print(f"2022 shape after recoding: {df_2022.shape}")
print(f"2023 shape after recoding: {df_2023.shape}")
print()
print("Columns:", df_2022.columns.tolist())

2022 shape after recoding: (442913, 30)
2023 shape after recoding: (431257, 30)

Columns: ['CHCCOPD3', 'INCOME3', 'EDUCA', 'SEXVAR', '_AGEG5YR', '_IMPRACE', 'EMPLOY1', 'MARITAL', '_URBSTAT', '_SMOKER3', 'EXERANY2', 'DRNKANY6', '_BMI5CAT', 'ASTHMA3', 'CVDINFR4', 'CVDCRHD4', 'CVDSTRK3', 'HAVARTH3', 'CHCKDNY2', 'HLTHPLN2', 'PERSDOC3', 'MEDCOST1', 'CHECKUP1', '_LLCPWT', '_STSTR', '_PSU', 'YEAR', 'COPD', 'DIABETES', 'PREDIABETES']


In [ ]:
# Missing data?

def missing_summary(df, label):
    print(f"\n{label} — missing data:")
    miss = df.isnull().mean().sort_values(ascending=False)
    miss = miss[miss > 0]
    for col, pct in miss.items():
        flag = "  ⚠ HIGH" if pct > 0.10 else ""
        print(f"  {col:<16} {pct*100:5.1f}%{flag}")
    print(f"  Rows with ANY missing feature: "
          f"{df.drop(columns=['_LLCPWT','_STSTR','_PSU','YEAR','CHCCOPD3'], errors='ignore').isnull().any(axis=1).mean()*100:.1f}%")

missing_summary(df_2022, "2022")
missing_summary(df_2023, "2023")


2022 — missing data:
  INCOME3           21.5%  ⚠ HIGH
  _BMI5CAT          10.9%  ⚠ HIGH
  DRNKANY6          10.4%  ⚠ HIGH
  _SMOKER3           7.9%
  HLTHPLN2           4.0%
  EMPLOY1            2.5%
  _URBSTAT           2.1%
  _AGEG5YR           2.0%
  CHECKUP1           1.3%
  MARITAL            1.0%
  PERSDOC3           1.0%
  CVDCRHD4           0.9%
  CVDINFR4           0.6%
  HAVARTH3           0.5%
  EDUCA              0.5%
  CHCKDNY2           0.4%
  MEDCOST1           0.3%
  ASTHMA3            0.3%
  CVDSTRK3           0.3%
  EXERANY2           0.2%
  DIABETES           0.2%
  PREDIABETES        0.2%
  Rows with ANY missing feature: 36.7%

2023 — missing data:
  INCOME3           19.9%  ⚠ HIGH
  _BMI5CAT           9.3%
  DRNKANY6           6.9%
  _SMOKER3           5.3%
  HLTHPLN2           4.3%
  _URBSTAT           1.9%
  _AGEG5YR           1.8%
  EMPLOY1            1.7%
  CHECKUP1           1.3%
  MARITAL            1.0%
  PERSDOC3           1.0%
  CVDCRHD4           0.9%
 

In [ ]:
# Cell 9: Drop rows missing required variables

def drop_required_missing(df, label):
    before = len(df)
    df = df.dropna(subset=["COPD", "_LLCPWT", "_PSU", "_STSTR"])
    after = len(df)
    print(f"  {label}: {before:,} → {after:,} rows "
          f"(dropped {before-after:,} missing outcome/weight)")
    return df

df_2022 = drop_required_missing(df_2022, "2022")
df_2023 = drop_required_missing(df_2023, "2023")

  2022: 442,913 → 442,913 rows (dropped 0 missing outcome/weight)
  2023: 431,257 → 431,257 rows (dropped 0 missing outcome/weight)


In [ ]:
#CELL 10: Final checks and save 

for label, df in [("2022", df_2022), ("2023", df_2023)]:
    print(f"\n=== {label} FINAL SUMMARY ===")
    print(f"  Rows:          {len(df):,}")
    print(f"  Columns:       {df.shape[1]}")
    print(f"  COPD cases:    {df['COPD'].sum():,} ({df['COPD'].mean()*100:.1f}%)")
    print(f"  No COPD:       {(df['COPD']==0).sum():,} ({(df['COPD']==0).mean()*100:.1f}%)")
    print(f"  Class ratio:   1 : {(df['COPD']==0).sum() / df['COPD'].sum():.1f}")

df_2022.to_csv(f"{PROCESSED}/brfss_2022_clean.csv", index=False)
df_2023.to_csv(f"{PROCESSED}/brfss_2023_clean.csv", index=False)

print("\n✓ Saved:")
print(f"  {PROCESSED}/brfss_2022_clean.csv")
print(f"  {PROCESSED}/brfss_2023_clean.csv")
print("\nNext: run notebooks/02_eda.ipynb")


=== 2022 FINAL SUMMARY ===
  Rows:          442,913
  Columns:       30
  COPD cases:    35,656 (8.1%)
  No COPD:       407,257 (91.9%)
  Class ratio:   1 : 11.4

=== 2023 FINAL SUMMARY ===
  Rows:          431,257
  Columns:       30
  COPD cases:    33,097 (7.7%)
  No COPD:       398,160 (92.3%)
  Class ratio:   1 : 12.0

✓ Saved:
  ../data/processed/brfss_2022_clean.csv
  ../data/processed/brfss_2023_clean.csv

Next: run notebooks/02_eda.ipynb
